In [ ]:

# ==========================================
# DECISION TREE CLASSIFICATION WITH ROC(general)
# ==========================================

# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    LabelEncoder, OneHotEncoder, label_binarize
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score
)

# ==========================================
# 2. Load dataset
# ==========================================

file_path = "PASTE_YOUR_DATASET_PATH_HERE"
target_column = "PASTE_TARGET_COLUMN_HERE"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print(df.head())

# Remove rows with missing target values
df = df.dropna(subset=[target_column])

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]

# ==========================================
# 3. Encode target column
# ==========================================

le = LabelEncoder()
y = le.fit_transform(y)

print("\nTarget Classes:", le.classes_)

# ==========================================
# 4. Identify numerical and categorical data
# ==========================================

numeric_features = X.select_dtypes(include=np.number).columns
categorical_features = X.select_dtypes(exclude=np.number).columns

# Handle missing numerical values
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Handle missing categorical values and encode
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# ==========================================
# 5. Split dataset
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================================
# 6. Create and train Decision Tree
# ==========================================

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("\nDecision Tree Training Completed!")

# ==========================================
# 7. Make predictions
# ==========================================

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# ==========================================
# 8. Evaluate model
# ==========================================

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(le.classes_)),
    target_names=[str(c) for c in le.classes_],
    zero_division=0
))

# ==========================================
# 9. Confusion Matrix
# ==========================================

cm = confusion_matrix(
    y_test, y_pred,
    labels=np.arange(len(le.classes_))
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Decision Tree Confusion Matrix")
plt.show()

# ==========================================
# 10. ROC Curve and AUC
# ==========================================

n_classes = len(le.classes_)

if n_classes == 2:

    # Binary classification
    fpr, tpr, _ = roc_curve(y_test, y_prob[:, 1])

    auc_score = roc_auc_score(y_test, y_prob[:, 1])

    plt.plot(
        fpr, tpr,
        label=f"Decision Tree (AUC = {auc_score:.3f})"
    )

else:

    # Multiclass classification
    y_test_bin = label_binarize(
        y_test,
        classes=np.arange(n_classes)
    )

    auc_score = roc_auc_score(
        y_test_bin,
        y_prob,
        average="macro",
        multi_class="ovr"
    )

    for i, class_name in enumerate(le.classes_):

        fpr, tpr, _ = roc_curve(
            y_test_bin[:, i],
            y_prob[:, i]
        )

        class_auc = roc_auc_score(
            y_test_bin[:, i],
            y_prob[:, i]
        )

        plt.plot(
            fpr, tpr,
            label=f"{class_name} (AUC = {class_auc:.3f})"
        )

    print("Macro ROC-AUC:", auc_score)

plt.plot([0, 1], [0, 1], "k--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Decision Tree ROC Curve")
plt.legend()
plt.grid()
plt.show()

print("\nFinal ROC-AUC Score:", auc_score)

# ==========================================
# 11. Display Decision Tree
# ==========================================

trained_tree = model.named_steps["classifier"]

feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

plt.figure(figsize=(20, 10))

plot_tree(
    trained_tree,
    feature_names=feature_names,
    class_names=[str(c) for c in le.classes_],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree Visualization")
plt.show()

# ==========================================
# 12. Feature Importance
# ==========================================

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": trained_tree.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance:")
print(importance.head(15))

importance.head(15).plot(
    x="Feature",
    y="Importance",
    kind="bar",
    legend=False,
    figsize=(10, 5)
)

plt.title("Decision Tree Feature Importance")
plt.ylabel("Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
